# Jumper su Kaggle — solo replay dei backup (nessun training)

Add Input: il Dataset `jumper-stairs-task` e l'output dell'ultima versione di `jumper-kaggle-stairs`. GPU T4 x2, Internet on. Dura ~20 min.

In [ ]:
# @title 1. Impostazioni (modifica solo questa cella)
import json
import math
import os
import re
import shutil
import signal
import subprocess
import sys
import threading
import time
import uuid
import venv
import zipfile
from collections import deque
from datetime import datetime, timezone
from pathlib import Path

# ── Cosa allenare ────────────────────────────────────────────────────────────
TASK = "jumper.stairs"                 # GPU 0: "jumper.stairs" (A, cieco, esportabile)
SECOND_TASK = "jumper.stairs_sighted"  # GPU 1: "jumper.stairs_sighted" (B, vede il terreno) oppure "" per nessuno
NUM_ENVS = 2048                        # per processo; su T4 16 GB sotto i 12 000 MiB di picco c'è margine
WORKFLOW_MODE = "Continue training"    # "New training" oppure "Continue training"

# ── Tempo ────────────────────────────────────────────────────────────────────
TRAIN_HOURS = 10.0      # training; poi ~1 h per backup, curve, replay, export (Kaggle taglia a 12 h). La prova: 0.5
SMOKE_ITERATIONS = 5    # mini-run di prova prima di quello lungo (0 per saltarlo)

# ── Warm start (New training, e le task in RESTART_TASKS) ────────────────────
WARM_START = True
WARM_SOURCE = "tasks/jumper/ref_free_jump/out/example/model_5999.pt"
# Task da far ripartire dal warm start anche in Continue (es. una policy che ha
# imparato una scappatoia del reward e non vale la pena continuare).
RESTART_TASKS = []
# Continue con codice della task cambiato: il manifest del backup viene riscritto
# con la configurazione attuale (osservazioni e rete devono restare uguali).
ACCEPT_CODE_CHANGES = True

# ── Sorgenti ─────────────────────────────────────────────────────────────────
REPOSITORY_URL = "https://github.com/KingKongRobotics/jumper.git"
SOURCE_REVISION = "b2e58669848a3ba4f209597cd6bebdfe9d0533a4"   # bloccato: la ripresa lo confronta
PATCH_DIR = "/kaggle/input/jumper-stairs-task"   # il Dataset con il codice della task (zip o cartella)
BACKUP_GLOB = "/kaggle/input/**/backups/*.zip"   # Continue: i backup delle versioni precedenti (Add Input → questo notebook)

# ── Solo replay: (task, riga del curriculum 0..9, secondi, etichetta) ────────
REPLAY_JOBS = [
    ("jumper.stairs", 0, 20.0, "A-fine-sessione1"),
    ("jumper.stairs_sighted", 0, 12.0, "B-scappatoia"),
]

# ── Replay ───────────────────────────────────────────────────────────────────
REPLAY_SECONDS = 20.0   # replay finale, per task
REPLAY_ROW = ""         # "" = la riga media raggiunta dal curriculum; oppure 0..9
REPLAY_WIDTH, REPLAY_HEIGHT, REPLAY_FPS = 960, 540, 30.0
# Camera: azimut 90 = di lato (0 = da dietro, 180 = di fronte), elevazione negativa = guarda in giù
REPLAY_AZIMUTH, REPLAY_ELEVATION, REPLAY_DISTANCE = 90.0, -8.0, 1.3
# Replay automatici durante il training (sulla GPU della task, dall'ultimo checkpoint
# completo): il primo dopo REPLAY_FIRST_HOURS, poi ogni REPLAY_EVERY_HOURS. 0 = mai.
REPLAY_EVERY_HOURS = 2.0
REPLAY_FIRST_HOURS = 0.5
REPLAY_PERIODIC_SECONDS = 10.0
REPLAY_MIN_FREE_MIB = 2500   # saltato se la GPU ha meno memoria libera di così

# ── Derivati ─────────────────────────────────────────────────────────────────
ON_KAGGLE = Path("/kaggle").is_dir()
WORK = Path("/kaggle/temp") if ON_KAGGLE else Path("/tmp/jumper-work")     # effimero
OUT = Path("/kaggle/working/jumper") if ON_KAGGLE else Path.cwd() / "jumper-out"  # persistente (output)
REPO = WORK / "jumper"
PYTHON = REPO / ".venv/bin/python"
STAMP = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
SESSION = OUT / f"session-{STAMP}"
BACKUPS = OUT / "backups"
TASKS = [t for t in (TASK, SECOND_TASK) if t]
for directory in (WORK, OUT, SESSION, BACKUPS):
    directory.mkdir(parents=True, exist_ok=True)
if WORKFLOW_MODE not in ("New training", "Continue training"):
    raise ValueError("WORKFLOW_MODE: New training oppure Continue training")
if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError(f"Jumper vuole Python 3.10-3.13, qui c'è {sys.version}")
for t in TASKS:
    if not re.fullmatch(r"jumper\.[a-z_]+", t):
        raise ValueError(f"task non valida: {t}")
print(f"{WORKFLOW_MODE}: {TASKS} x {NUM_ENVS} env, {TRAIN_HOURS} h di training. Output in {OUT}")


In [ ]:
# @title 2. Setup: GPU, clone, venv, torch, EGL, codice della task  (~6-10 min)
def run(command, *, cwd=None, check=True, env=None, timeout_s=None, tail_lines=80):
    """Esegue e mostra l'output in diretta. Se scade `timeout_s`, SIGINT a tutto
    il gruppo (training e helper salvano e chiudono) e si rientra con quel codice."""
    print("+", " ".join(str(p) for p in command), flush=True)
    command = [str(p) for p in command]
    tail = deque(maxlen=tail_lines)
    process = subprocess.Popen(
        command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True,
    )
    timer = None
    if timeout_s:
        timer = threading.Timer(timeout_s, lambda: os.killpg(process.pid, signal.SIGINT))
        timer.daemon = True
        timer.start()
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            tail.append(line[-4096:])
        code = process.wait()
        if code and check:
            raise subprocess.CalledProcessError(code, command, output="".join(tail))
        return subprocess.CompletedProcess(command, code, stdout="".join(tail))
    except BaseException:
        if process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGINT)
                process.wait(timeout=30)
            except (ProcessLookupError, subprocess.TimeoutExpired):
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
        raise
    finally:
        if timer:
            timer.cancel()
        process.stdout.close()


# ── GPU ──
run(["nvidia-smi", "-L"])
rows = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total", "--format=csv,noheader"], text=True,
).strip().splitlines()
if not rows:
    raise RuntimeError("Nessuna GPU: Settings → Accelerator → GPU T4 x2")
gpus = [[p.strip() for p in r.split(",")] for r in rows]
capability = float(gpus[0][2])
if capability < 7.5:
    raise RuntimeError(f"GPU {gpus[0][1]} (compute capability {capability}) non supportata: scegli T4 x2, non P100")
GPU_NAME = gpus[0][1]
if SECOND_TASK and len(gpus) < 2:
    raise RuntimeError("SECOND_TASK vuole due GPU: scegli T4 x2 oppure metti SECOND_TASK = ''")
print("GPU:", [g[1] for g in gpus])

# ── Clone al commit bloccato ──
if not REPO.exists():
    run(["git", "clone", REPOSITORY_URL, REPO])
run(["git", "fetch", "origin", SOURCE_REVISION], cwd=REPO)
run(["git", "checkout", "--detach", SOURCE_REVISION], cwd=REPO)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()

# ── Il codice della task, dal Dataset, sopra il clone ──
patch = Path(PATCH_DIR)
if not patch.exists():
    # Kaggle monta i dataset sotto /kaggle/input con un percorso che è cambiato
    # nel tempo (<slug>, oppure datasets/<utente>/<slug>): cerca il file guida.
    import glob as _glob
    hits = _glob.glob("/kaggle/input/**/tasks/jumper/stairs/env_cfg.py", recursive=True)
    if not hits:
        found = sorted(str(p) for p in Path("/kaggle/input").rglob("*") if p.is_dir())[:40]
        raise FileNotFoundError(f"codice della task non trovato sotto /kaggle/input. Cartelle viste: {found}")
    patch = Path(hits[0]).parents[3]
    print("Codice della task trovato in", patch)
zips = sorted(patch.rglob("*.zip"))
if zips:
    with zipfile.ZipFile(zips[-1]) as z:
        names = z.namelist()
        if any(n.startswith("/") or ".." in Path(n).parts for n in names):
            raise ValueError("zip con percorsi non sicuri")
        z.extractall(REPO)
    print("Applicato", zips[-1], f"({len(names)} file)")
else:
    copied = 0
    for src in patch.rglob("*"):
        if src.is_file():
            rel = src.relative_to(patch)
            # la cartella del dataset può avere un livello in più (nome del dataset)
            parts = rel.parts
            if parts and parts[0] not in ("tasks", "tools", "notebooks", "docs"):
                rel = Path(*parts[1:])
            dst = REPO / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            copied += 1
    print(f"Copiati {copied} file da {patch}")
for required in ("tasks/jumper/stairs/env_cfg.py", "tools/warm_start.py"):
    if not (REPO / required).is_file():
        raise FileNotFoundError(f"manca {required}: il Dataset non contiene il codice della task")

# ── Continue: quale backup per ogni task, e le sue versioni dei pacchetti ──
import glob

def backup_manifest(archive):
    with zipfile.ZipFile(archive) as z:
        info = z.getinfo("run/colab-run.json")
        if info.file_size > 1024**2:
            raise ValueError("manifest troppo grande")
        return json.loads(z.read(info))

BACKUP_FOR = {}
CORE_VERSIONS = {}
if WORKFLOW_MODE == "Continue training":
    candidates = []
    for archive in sorted(glob.glob(BACKUP_GLOB, recursive=True)):
        try:
            m = backup_manifest(archive)
        except Exception as exc:  # noqa: BLE001
            print("salto", archive, "->", exc)
            continue
        cfg = m.get("config", {})
        if m.get("schema") == 1 and m.get("checkpoints") and cfg.get("task") in TASKS:
            candidates.append((Path(archive).stat().st_mtime, archive, cfg))
    for t in TASKS:
        if t in RESTART_TASKS:
            continue
        mine = [c for c in candidates if c[2]["task"] == t]
        if not mine:
            raise FileNotFoundError(f"nessun backup di {t} in {BACKUP_GLOB}: Add Input → la versione precedente del notebook")
        _, archive, cfg = max(mine)
        BACKUP_FOR[t] = archive
        print(f"   backup del {datetime.fromtimestamp(_).strftime('%Y-%m-%d %H:%M')} (ora locale del server)")
        if cfg["revision"] != SOURCE_REVISION or cfg["num_envs"] != NUM_ENVS:
            raise ValueError(f"il backup di {t} è del commit {cfg['revision'][:9]} a {cfg['num_envs']} env: "
                             f"SOURCE_REVISION e NUM_ENVS devono coincidere")
        if CORE_VERSIONS and CORE_VERSIONS != cfg["runtime_versions"]:
            raise ValueError("i due backup hanno versioni dei pacchetti diverse")
        CORE_VERSIONS = dict(cfg["runtime_versions"])
        print(f"Continue {t} da {archive}")
    # Nota: `pip install -e .` risolve torch oltre la 2.9.1 (misurato: 2.14.1 dal PyPI);
    # in Continue si reinstalla esattamente la versione del backup, torch compreso.

# ── venv isolato ──
if not PYTHON.exists():
    venv.EnvBuilder(with_pip=False, system_site_packages=False).create(REPO / ".venv")
run([sys.executable, "-m", "pip", "--python", PYTHON, "install", "--upgrade", "pip", "-q"])
try:
    run(["apt-get", "install", "-y", "-qq", "libegl1", "libgl1", "ffmpeg"], check=False)
except Exception as exc:  # noqa: BLE001
    print("apt-get non riuscito (continuo):", exc)
TORCH_INDEX = "cu126" if capability < 8.0 else "cu128"
run([PYTHON, "-m", "pip", "install", "-q", "torch==2.9.1", "torchvision==0.24.1",
     "--index-url", f"https://download.pytorch.org/whl/{TORCH_INDEX}"])
run([PYTHON, "-m", "pip", "install", "-q", "-e", str(REPO)])
pins = [f"{k}=={v}" for k, v in sorted(CORE_VERSIONS.items())]
if pins:
    run([PYTHON, "-m", "pip", "install", "-q", *pins])   # le versioni esatte del backup, come il notebook ufficiale
run([PYTHON, "-m", "pip", "install", "-q", "matplotlib"])

# ── Rendering headless per il replay MP4 (il training non ne ha bisogno) ──
# Il notebook ufficiale esige l'EGL di NVIDIA; l'immagine Kaggle non lo espone
# (misurato: nessuna libEGL_nvidia nei percorsi standard). Si cerca ovunque e,
# se manca, si ripiega sull'EGL software di Mesa: basta per 12 s di video a un
# solo robot. Il replay è comunque in try/except: non blocca mai il backup.
import glob as _glob
os.environ.update(MUJOCO_GL="egl", PYTHONUNBUFFERED="1", MPLBACKEND="Agg", PYOPENGL_PLATFORM="egl")
_egl_hits = []
for pattern in ("/usr/lib64-nvidia/libEGL_nvidia.so.*", "/usr/lib/x86_64-linux-gnu/libEGL_nvidia.so.*",
                "/usr/lib/x86_64-linux-gnu/nvidia/current/libEGL_nvidia.so.*", "/usr/lib/nvidia/libEGL_nvidia.so.*",
                "/usr/local/nvidia/lib64/libEGL_nvidia.so.*", "/usr/local/cuda*/lib64/libEGL_nvidia.so.*"):
    _egl_hits += [h for h in _glob.glob(pattern) if Path(h).is_file()]
if not _egl_hits:
    try:
        _egl_hits = subprocess.run(["bash", "-lc", "find / -xdev -name 'libEGL_nvidia.so.*' 2>/dev/null | head -5"],
                                   capture_output=True, text=True, timeout=120).stdout.split()
    except Exception:  # noqa: BLE001
        _egl_hits = []
if _egl_hits:
    lib = Path(_egl_hits[0]).resolve()
    (WORK / "nvidia-egl.json").write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": str(lib)}}))
    os.environ["__EGL_VENDOR_LIBRARY_FILENAMES"] = str(WORK / "nvidia-egl.json")
    os.environ["LD_LIBRARY_PATH"] = ":".join([str(lib.parent), *[e for e in os.environ.get("LD_LIBRARY_PATH", "").split(":") if e]])
    print("EGL NVIDIA:", lib)
else:
    print("EGL NVIDIA assente: il replay userà l'EGL software di Mesa (più lento, ma basta per un video breve)")
    try:
        run(["apt-get", "install", "-y", "-qq", "libegl-mesa0", "libgl1-mesa-dri", "libgles2"], check=False)
    except Exception as exc:  # noqa: BLE001
        print("apt-get mesa non riuscito (continuo):", exc)
    os.environ["EGL_PLATFORM"] = "surfaceless"

run([PYTHON, "tools/colab.py", "check", "--repo", REPO, "--out", SESSION / "runtime.json"], cwd=REPO)
runtime = json.loads((SESSION / "runtime.json").read_text())
for pkg, want in CORE_VERSIONS.items():
    if runtime["versions"].get(pkg) != want:
        raise RuntimeError(f"versione di {pkg}: il backup vuole {want}, installata {runtime['versions'].get(pkg)}")
listing = run([PYTHON, "scripts/train.py", "--list"], cwd=REPO).stdout
for t in TASKS:
    if t not in listing:
        raise RuntimeError(f"{t} non compare in --list: manca la riga di import in tasks/__init__.py del Dataset")
run([PYTHON, "scripts/train.py", "--task", TASK, "--dry-run"], cwd=REPO)
print("\nSETUP OK —", GPU_NAME, "| commit", COMMIT[:9], "| versioni", runtime["versions"])


In [ ]:
# @title 3. Solo replay: ripristina i backup e registra i video (nessun training)
from IPython.display import Video, display

RATES, RESTORED = {}, {}
for t in TASKS:
    dest = WORK / f"restored-{t}-{uuid.uuid4().hex[:8]}"
    info = SESSION / f"restore-{t}.json"
    run([PYTHON, "tools/colab.py", "restore", "--archive", BACKUP_FOR[t], "--destination", dest, "--out", info], cwd=REPO)
    RESTORED[t] = Path(json.loads(info.read_text())["checkpoint"])
    rf = SESSION / f"rates-{t}.json"
    run([PYTHON, "tools/colab.py", "rates", "--repo", REPO, "--task", t, "--model", "jumper", "--out", rf], cwd=REPO)
    RATES[t] = json.loads(rf.read_text())
    print(t, "->", RESTORED[t])


def train_env(gpu_index):
    e = os.environ.copy()
    e["CUDA_VISIBLE_DEVICES"] = str(gpu_index)
    return e


VIDEOS = OUT / "replays"
VIDEOS.mkdir(parents=True, exist_ok=True)
for task, row, seconds, label in REPLAY_JOBS:
    steps = math.ceil(seconds * RATES[task]["control_hz"])
    it = re.search(r"model_(\d+)", RESTORED[task].name).group(1)
    video = VIDEOS / f"{label}-{task}-it{it}-row{row}.mp4"
    cmd = [PYTHON, "scripts/play.py", "--task", task, "--model", "jumper", "--checkpoint", RESTORED[task],
           "--backend", "warp", "--device", "cuda:0", "--num_envs", "1", "--headless", "--steps", steps,
           "--physics-hz", RATES[task]["physics_hz"], "--video", video, "--video-fps", REPLAY_FPS,
           "--video-width", REPLAY_WIDTH, "--video-height", REPLAY_HEIGHT, "--strip-visual", "off",
           "--video-azimuth", REPLAY_AZIMUTH, "--video-elevation", REPLAY_ELEVATION,
           "--video-distance", REPLAY_DISTANCE, "--stairs-row", row, "--measure",
           "--replay-info-out", VIDEOS / f"{label}-replay-info.json"]
    try:
        run(cmd, cwd=REPO, env=train_env(0))
        print("VIDEO", video, f"({video.stat().st_size / 1e6:.1f} MB)")
        display(Video(str(video), embed=True))
    except Exception as exc:  # noqa: BLE001
        print(f"[replay {label}] fallito: {exc}")
print("FATTO: i video sono in", VIDEOS)
